# 📋 STR01 · 미션 04 | 작전 템포 변동 및 기상 악화 연계 부대별 결핍 위험 일자(Stockout Date) 정밀 예측 고도화

**학생용 Student · 직접 실습**  
CMS 편성: 3시간 · 교육용 가상 데이터 사용

---

## 📖 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **작전 템포 변동 및 기상 악화 연계 부대별 결핍 위험 일자(Stockout Date) 정밀 예측 고도화** 과제를 수행합니다.

실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 사전 계획 템포, 다음 7일 악천후 예보, 최근 소모 추세, 보급 후 경과일 같은 파생 피처 13개를 설계하고 이유를 한 줄씩 설명한다.

릿지 회귀 기준 모델과 단순 DOS 방식을 MAPE로 비교하고, 계획 템포 +30%와 악천후 50%를 가정했을 때 예측이 얼마나 바뀌는지 시나리오로 확인한다.

**만들 결과물:** `src/features/str01_feature_engineering.py`, `reports/str01_feature_report.md`, `outputs/M04/results.json`, `forms/STR01_M04_forms.md`

## 📌 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

> **보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다.

- 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다.
- 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다.
- 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요.
- 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'STR01_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 2개: STR01_M04_forms.md, STR01_M04_lesson.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAAIAAAAIVxyZromsQgAAJkUAAASAAAAU1RSMDFfTTA0X2Zvcm1zLm1k3Vhda1vZFX3XrzgQCi21LTmZaRlDWgJTyECHlmbeSjEmo0zTSWxjKbQF'
    'P8jxVVAkBSsTXfvKuRLSRP5QquBrWbZkUFrwT8m83XPuf+ja+5wjXTnufBT6MCUmts7nPnuvvfbauibufPaH1Lz4NPWBUI3nyjkWavu5KtXFu1yVR5qOiPJ+'
    'tNUV8iQnt2pCBlsiHAZqc0MoNx/VsGwnCE8cIc9yspyT+CvsBZH7TCjfiXY8oeoj1aj89E525e6XK4+y4uOlbPpnQjVdGeSE8gpq2BLhSVNuOTgskVAvK/Jk'
    'KKKtp2GQWxB/VE4z2qz/SUTlstqsi6g6Uj3vYkAmtHNCvj6Uz4aCzZ1aIeZTYeDj8j4sVXVHYArPi1wPd4np3Z/e+v1vxPUPf0KrsUDujYQsdWn1dkH5I9kJ'
    'ZuypymnjrbBMhKeBKmCg1JbdvtqpKMcX8nFHvqgLTMviXuRiID+S+106MAy2ZbGKmblEAp4Me2/x6JFq+gti9cHS8i8vBvRrcW0pe38Fj+9G7mESs6o2NFbS'
    'G8Kgpuo5Ebk1VTy9GKiSTw7GY8LAsSacO+FpJXI7slSQpfbctFdU3TOOgOG+alZgg1CPuxRmvCvafYZTaw4+IjDypE9PQaTobvk6T/eND04krl275HLtnERi'
    '3Q6s06HqcXAxkKVDwAEDJuD6dhHVXHMLHCbCwSg8DrBIFhz49mIAFxKy1nHi7OysuOJ/zIjJz4/mE/lO+3vsu6ZPbwecEFJyoY46BbkdbT/FNh315FTQ4dgz'
    'BysxzSCGf5tO9BJoDw4BZuzm7OE08Jvf4UiKUcEXH//uzrThhIZeX8j9f6mD3PSUeYMF6nsr6J23PiFjGVrISBn4+N3Lqf06DctWPZGYtRCMyhUYMWYixsiC'
    'eLfdokNk6RSJAxrqYx8ZxRNmVO30ZaMs+Om8G4fe+gQ5IeRBKxwOsSBPeVrLGV8s0K3w+clp5DaxpyJfleGut1X5AgbSg9y3nBwvK0QJxSqxGgZoH/hJ1fHT'
    'ZzRvblCm61k+33XUwQbHb69PG/G4yH2zwEEt54gx5euOam5QtEptXCXUblX13tBdsgs3ehXe1sypxh6xqDr3Jmvtg791rbXQE7JTZXz8lymU0KZRAtO1xD7W'
    'ZjaAmeGkr1r9GOVoLwpDc8ZEG3P2E1Ns4CmPfUcH1BzjBdQBA4w416wTAsAvYKHxm/UtBuHrkxhywoA88mVVbEfFYYxXxm7RSOMAm9SA85pAL/YXPFAeXnzS'
    'l3vs+FNffZ039jLl1UdYZnz9nx17lXvf8/CVA4kb4XE/VrsAG9UK4h7mtwrZK0TekJypgn9qJ28S5sPBELZRLsqDCtW7yO3L0jnqU1/tdnCufOzZOFqGNjwz'
    'iQ8QCr6JRUDHk8GO+BT34OJyWdZH8mgYOdq9HHBKTZtIskfFZDLHCXp5znJW3IIY7rX7vzeTfW+3EzkZtaH1AF9rbgMBnPlcyliIwINvkLVU0fTS4CvLvaxF'
    'wMWTFNUvwPgYd6wmsOFoiD/JbxVADIqpQEev2wT4n2Dp/WfP8iP7hXEgzEfKsH5BuQWT0SYEzJQ2ZPkhEkqebXB6AFmQHPTOodzHe2XLx5/Gd7HdAJCqA61H'
    'Q5X3SFC1kPOOR7Ss9QVVBEuuUC9hP0d0ET0BMYNdN31dA5AFxPz8Z5wxeEQVz5Ha3+SqskRQR4LgXVCeJd/G4ZucqzWLe6pKXdzTjTxnXIOovh6SoCMiycsm'
    '0I1H4DdpFJxM471/CEsumiREDLYT7g38sNelNFFeGwwyXmzh8K1U/HDpy/RiZunh6oN0BsP8D4rxaGjspdvKh8x/zjGLJotHrkeuI0u1MUuQwc4xapwxkgF+'
    'Rip0Kh95jnnmknjU/qr50P2k69WBw4+/VL7XzRLSKNpXRl5PEoJfPuFME1YG0XcAHzPzpG7QoGixsXBlW0Ivs10JK9fpCjOlUxl+P6RZYTZk3WqlQLx9YMcd'
    'lOWLJrmM6DeCAtitMgIMlMZqjWUwHAG9s9uxuncixePRR5nCdgNCYmn3LfcxXARNHzMuaFpA6ap2MYDygacbjg2V9hx3ELqybhd0M/JrA7B1cd06mWG2IDJr'
    'd5P30kvZR2vpTDKTXUvNL5qPi+nlL+4vp9Nr95e/mFv9O1XTuNlwllYj2kRtOPLy85kHKzevz8/8+f7N+esfindPvqIGiBIC6Q7XaqfE+hCiBgpnrB3hyLGr'
    '539B3Z1paiC6IHOizZx8FlAL5wBqt+fEbQpVtL1HGGfViCTdeUJMhGKKAsvN0JMyUE1RR46iLnYC2xTq0LGCdF5RKVVuEQdyYmVTP7+tiwJJcgMHHCK4tPYp'
    'BaLnPoIDY80ioIG2goQQBnXWUa1jAyZkqU5Syy2aRlElnvtJTa3YYkTWJHD00JfOb9OZzMoyAYDTWmc13cCJp2kbbzCZ8H7Yb9iwM07+H9ttw0HT/RLAxhDQ'
    '3HxOlKBrsjFYV8SkJu4xPxp6nHjxkgtuwAXcAeiGSSNIMFTZptMubo+35pfh81EK/4fnZYZb04XmJlInUsWHio8sIuS9e9r+KAUgTSCm6SSuD7lJJLWeTf0K'
    'p8ISM3P7Zkow9sffvUS75XCQ416BfUOQR56wGwDno6GlyHDgq/Oard9Yu9+NXUlfX9CppHH2u8iWMp/QKDBhmj6x5EN5YhqVcUb89W+L9+5yBrAI43S84ruH'
    'ie22A8INDW9M9zTIkANvma8VTO5qS9WLYNJ/2DGo1sBDK4Z4m9yeaF0WQq9QB/aSOhtusMMcXcUoO1mb2OoFDDW6GiNXJNgHJsGwEIr9B/DqjFhLr66sZS+v'
    '0qNzDz+fEShQq4+wAOcncdyjB9nM3F9ABzPi3sraw0ySv9tbxOwif8YeIgrNDNwka+n3dV6rWe5NNc/oQm2/PtCam8KxEwALsuLZV1pBS9Ju2NJtE5pCx6cB'
    'e4W+kPWZlnBcIlzAOmoUAGIC62XH/RtQSwMEFAAAAAgAAAAhXKlBrxIdCgAAQhoAABMAAABTVFIwMV9NMDRfbGVzc29uLm1k7VhdT1vZFX33rzhSFKnVuGDD'
    'pNMgpaOoiZqojRpN+tSqAjc4KR0CCDvt9M3AhRrsFCf4gk2uXdNAwKmrXoyNjWRUiZ+Sebvn3P8wa+9zrn2d0A/eqtG88HE/ztl7r7XXXudeE49++UUsLh7E'
    'PhXvM0Whqi9VzRL+quNvNoQ8ycjNspDupvC6rlpZEspe9ct4bMf1TiwhTzMyn5H4y2u6vv1CKMfyd0pCVXqqWvjeo/T84y/nn6fFnUQ6+X2harZ0M0KVsqq7'
    'J7yTmty0sFgkcu2awNp+ue6d9IR8V/dfOpGI52Zow589vI9tLGVVhMy1ZO4MC8s9R1lOaHdVbWC5i45ay8t3RxcdvYa484tHqpwJYvOa5/KkJQ96Zj3frstc'
    'Vub2Ry7NmmIJki5lgzfdkiot+bYj/HxerVSEX+ypZgkBXa0WtJaug2+X5EaRd9rPIPkj+aKrKijtYV5u1Sg6XZ/XBXnSFSpb8rfXhWwWsTrl5m+uo1CRyA/6'
    'T1g1H3HJf3bVagmZ1SzVbAmV21cbbbVTQN1oO73OoAL991HDlaz46Uxa+H/pKutYjKuc47nW15kirdHd89xXqoIFbUvmSl9n7NDOvOaEUMsNKib44e++MDWN'
    'Arx9VcmLz1AOIpFqOv5ry1Q2KlTb8Tq4wfgJdVpUVhfvnLS87rqgBwEeAKGXPbesKpnh+ov4uOc6XFRrnza2S+ADSt9STo0uE2Jq31LFOj0h363iAiMv3/5L'
    'HWK13bzXyQyjwGTMHaksEwnQ11WuQtA8uP3wLiN+Znntgt4rOpyv+GQ8dp3wGaR6I3adIiFa12x/O8sg23lDA67pdk8eZuVyCXsVZSYPoCg2AIBr8m1DlfZp'
    'W79sq0rXJIDy/1pD/pvhklx0hpIR3N0fFC2GonGRVpYoGtwC4qZ2w29TymLsxnV6mhhLrZBr0NNIxOnJuhsNVtUIIE3htV1wgrgnGy3NPSGX63KrQlWQGweE'
    'ilztITVdmW1kjDsjIU4tu+rUkY3ehEgtPh59kkykny8mU6Op9GIsPmn+nUzOPZ2ZSyYXZ+aejiz8qS9X/fB1B9Uc76QtSD7e5Klh8J+/1ubeB8+Vjcq3Ljo/'
    'mZ9N/JZipj4LLur2Qaxdf6MrT5eCK37RIi2qVXBZ/Or+Q478vCi3nAkx9T8HPDUCfrZVroFfoBtX12s3/JKFfWoZVT0I+Nz8exAZMer2fWbYKdWMRAs9MtTR'
    '8oVLcFnuhBgKdGo6kU6MfnH39p0Hd0eeTU8xmkc9bAV4z5GWWQIYYO+bRBIf7b7lQFiLtlotSisLzOLxoTuaKKq6KmsH8R/iFoLXoSFWdPKPTIciM+jvsPTk'
    'HKAA/r9Eh6FyPJImMZImn8wvPktRjAzq9Pzj1OhicmF+MZ1SlRKLXNli9DSIfg7k0WhXWrKZNTJIz5oK67oS/NQLld6VcKK3GCG89SzxZXIylXi2MJtMTTGb'
    'LGWfT4jp6Oz8rbF49Hczt+JjN8T7tVe41fOOXeG1Mmh0VCWsj2hTPecoEWi1njQkLqz/XEjTWQS2rBf9lUwf2IvOvRFxjwTd3z4gShjFVDtrJLZqz/WOW5Q9'
    'hFUdWjSKsiUqTd0VQ4ChO4W03rDc2BtYkMUxHfvkHgNol4n+uqdYwLHEMhbuCoxZ9CmCNQ8NZAmjUqLf+rEaeQk4TkPIdCQ3DMURTGNzHaqUhY5phpF7QCas'
    '8W4Xmfl2SzyIU3J/SMzOgNLJydnk9NPk4qB9hkg2RADuH24n65j34q5iST5pqb1WSAT3WsMtRnAiQg7ETNEWjVkuv5kIA2MBmag56FQBbwCvgz456VLdMO2Y'
    'E82Mv0bGBdJ+UKIFkWZobsnlI+9sKUiCiG2w8MsOPApduWb+jrPCa1bR6DI08/MFDLFIxAiqdkY8HFhfufbYn1WRc6qcc+ZmmhlNrzmq3OUq7ZQCJSdN/zOV'
    'DM/TSLNr3ln+c562vGeoDFAqItjhntftGvMS7EkS56BzgDPPNZqzsmnzPKJF66Hkg1THOFUjf5yBwQauUeu+vsr5MLIBnk3Y1h4X1sCMN1/tm6XwLnLmjJgk'
    'FLOqLandeiiVIRJMiIXZxNxnFx36NbmYSM/Mw5E1QPFRU7KBoQt8C+NHA8Qhd4g94a6Ckn5In6F5zX2se4gEs1YIoRyISVhFjNVgUr9bpf0uLea45o3mI1iK'
    '1uK6Vhu6dKCOYeWgpkNDKXgwHPcHgqEractcm3wOAY1/CiUjN1wW1hF2QVaV+G9usTtGIp9TD2nrhiw15tr3jEPiwiYmYBXHxB6OczLREu5lFCTXZpdvsS/N'
    'DUzVZRUyJ6O2o/62aritS2+8Sb+5tBk3aVOrM1WMNvRb7RJJaFELWaa2DlYNSwhK9dd1Y7eiPNFgF2s2/mx0iTV69vFxpV3AkBFq44waiY5UK0MjtzWcBDPj'
    'EoFp4XAEhWhm1NuKUG/guoF/IRK5xHIzU8wRgweJ4Il12Zz/eIrcjOEnNIOx/ogdDo4cNIDer+/fjOGgMZg0xPmhntS9RDRKx36MVQej5N6tmOAR2D92aqfP'
    '7uHUQncyNXfWzHEB1e0fFzuOOiuTueXDnQXHE9qSGp5WxQu4AarmdS9k+TCnxTNs3KPij19NPnnMY4/Jwt18SbeGxqAmCu1QLfWPonSRiU4CxTIQjHAdqdpy'
    'OQRGL7hW6WEag2togqCx+i4vMMdgHbO6bKF/DErkk67mj6aMPfvgQX2VjBw9g2PwwnM8A4cHO5d6PptOjfw+NT9HN9nxGU6jTNHQCFW7RdX8Bzqo8hJc4UOj'
    'amXxf1ifooFDRuJeB+dZ/JBvezRmQiM6fMTQ/aUTNsd3UndyW8wQHpYQaW3VqPEIJ+sYOJhGMupJi+oWZHB13HrCh5oRW0B3CBaawXvhzxDUUNS4DAGLAXey'
    'QRENwCJ+GYTG9HJ3G3Jp7aKEzGANII2P8NceTdKJ/+9vH9Sd8H9Ank/wTPQBHqRh7CRYEMyhMjQcwx6dRKy6SoINkdJmnceF+RpghgN5M3qSp6oG96IDd+BY'
    'qmoFmqgrxzNdI7idNUMqMmZqy/7hCmdWAiwcLWoUHrI6Xgjid6cLehXTANVXp3W1d2w4BC+vrbzufnDDtD+WGNUSgFeM6+vjxZ8zXls/T6agPoQ7G309t2kH'
    'bmFzPtkpmAYYoD1u0GZWTHwLP8QYEev3m9FDjbwW3jMSAHQKm3VtCFmWR7Uq9w9WxtsOSv+dm/g2ugme07CM1YPRgT3HBnpCUlOaE6824nQKDI4Pg776VPeV'
    'noNXUNGo+C/uIyr+nfWIauMx+vGnJ5IFrQPsSRAxT1nPfYUk2MFrVaEZcYZu12nrIzyhsOOCAjjsBLwnBHINvBsYefzJRp4uBFvoDXn6Yw8aMzQQbLDZr2aN'
    'ERjUKxL6dv8fPt2P6gX1F/z+B/wR85VDn2iRrf5QQgq8W9cXSVb1B4INHN1rAD10UPoGUEsBAhQDFAAAAAgAAAAhXHJmuiaxCAAAmRQAABIAAAAAAAAAAAAA'
    'AKQBAAAAAFNUUjAxX00wNF9mb3Jtcy5tZFBLAQIUAxQAAAAIAAAAIVypQa8SHQoAAEIaAAATAAAAAAAAAAAAAACkAeEIAABTVFIwMV9NMDRfbGVzc29uLm1k'
    'UEsFBgAAAAACAAIAgQAAAC8TAAAAAA=='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 04. 작전 템포 변동 및 기상 악화 연계 부대별 결핍 위험 일자(Stockout Date) 정밀 예측 고도화

**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 작전 템포 변동 연계 부대별 결핍 위험 일자(Stockout Date) 정밀 예측을 고도화한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **작전 템포 변동과 기상 예보를 반영한 파생 피처로 부대별 결핍 위험 일자(Stockout Date)를 예측하는 기준 모델을 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **사전 계획 템포, 다음 7일 악천후 예보, 최근 소모 추세, 보급 후 경과일 같은 파생 피처 13개를 설계하고 이유를 한 줄씩 설명 수행하고 검증하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/features/str01_feature_engineering.py 및 기준 모델` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 실무에서 납득 가능한 피처 설계의 논리성

#### 미션 PASS 조건

- 정량 기준: 파생 피처 10개 이상 생성, 기준 모델 MAPE 25% 이하 달성

- 진행 기준: 피처 파이프라인 및 베이스라인 모델 성능 지표 도출

- 검증 방법: 교차 검증(Time-series Cross Validation)을 통한 과적합 방지

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


### AI 활용 예시 · 미션 04 전용

> 나는 Python을 처음 배우는 학습자야. **작전 템포 변동 및 기상 악화 연계 부대별 결핍 위험 일자(Stockout Date) 정밀 예측 고도화**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: src/features/str01_feature_engineering.py / make_samples: d,lo=21,hi=125 → 과거 관측·사전 계획·가상 완전예보로 만든16개 피처와 메타데이터·H. H는 향후 보급 없음 조건의 소진일수이며 모델 입력에 넣지 않는다. t0+H가 학습 기준일 이내인 표본만 학습한다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.


In [ ]:
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """작전 템포 계획과 기상 예보를 포함한 파생 피처 13개를 설계해 릿지 회귀 기준 모델로 부대별 결핍 위험 일자(Stockout Date)를 예측하고, 단순 DOS 방식과 MAPE를 비교하며 템포 상승·악천후 가정 시나리오 영향을 확인한다."""
    import numpy as np
    import pandas as pd
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
    # 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
    # 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
    # ---------------------------------------------------------------------------
    ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
    N_UNIT, DAYS = 20, 160
    UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


    def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
        """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
        rng = np.random.default_rng(seed)
        rows = []
        for u in range(1, n_unit + 1):
            s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
            surge = np.zeros(days)
            for st in rng.choice(days - 10, 4, replace=False):
                surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
            tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
            plan = tempo + rng.normal(0, 0.08, days)
            wx = (rng.random(days) < 0.15).astype(int)
            for it, (b, sens, tg) in ITEMS.items():
                stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
                for t in range(days):
                    c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                    r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                    stock = max(stock + r - c, 0.0)
                    rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
        return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


    def make_samples(d, lo=21, hi=125):
        """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
        out = []
        for (u, it), g in d.groupby(["unit", "item"]):
            c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
            pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
            cs = np.cumsum(c)
            tg = ITEMS[it][2] * ITEMS[it][0]
            for t0 in range(lo, hi):
                idx = np.searchsorted(cs, cs[t0] + stk[t0])
                if idx >= len(cs) or idx <= t0:
                    continue
                c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
                last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
                out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                            "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                            "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                            "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                            "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
        return pd.DataFrame(out)


    def mape(a, b):
        # [TODO : 학생 실습 4] 작전 템포 변동 및 기상 악화 연계 부대별 결핍 위험 일자(Stockout Date) 정밀 예측 고도화 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    d = make_data()
    s = make_samples(d)
    X = [c for c in s.columns if c not in ("unit", "item", "t0", "H")]
    DESC = {"stock": "현재 재고", "c3": "최근 3일 평균 소모", "c7": "최근 7일 평균 소모", "c14": "최근 14일 평균 소모", "dos7": "재고/7일 소모(DOS)",
            "dos3": "재고/3일 소모", "trend": "7일/14일 소모 추세", "burst": "3일/7일 소모 급증도", "plan7": "다음 7일 계획 템포 평균", "plan_ratio": "계획 템포/현재 템포",
            "tempo0": "현재 작전 템포", "wx_fc": "다음 7일 악천후 예보 비율", "since_rs": "마지막 보급 후 경과일", "rs_avg": "최근 14일 평균 보급량", "fill": "목표 재고 대비 현재 재고", "item_code": "품목 코드"}
    derived = [c for c in X if c not in ("stock", "tempo0", "item_code")]
    print(f"[피처 설계] 전체 {len(X)}개 중 파생 피처 {len(derived)}개 [교육용가상자료]")
    for c in derived:
        print(f"  - {c}: {DESC[c]}")

    tr, te = s[s["t0"] <= 90], s[s["t0"] > 90]
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(tr[X], tr["H"])
    p_base = te["dos7"].values                                           # 단순 DOS 방식
    p_ridge = np.clip(model.predict(te[X]), 1, None)                     # 기준 모델
    y = te["H"].values
    res = pd.DataFrame({"모델": ["단순 DOS(Baseline)", "릿지 기준 모델"], "MAPE": [mape(y, p_base), mape(y, p_ridge)],
                        "MAE(일)": [np.abs(y - p_base).mean(), np.abs(y - p_ridge).mean()], "RMSE(일)": [np.sqrt(((y - p_base) ** 2).mean()), np.sqrt(((y - p_ridge) ** 2).mean())]})
    print(f"\n[예측 성능] 학습 {len(tr)}건(~90일), 시험 {len(te)}건(91일~)")
    print(res.round(3).to_string(index=False))

    # 결핍 위험 일자 예측 표(100일 기준)
    now = s[s["t0"] == 100].copy()
    now["예측H"] = np.clip(model.predict(now[X]), 1, None)
    now["결핍예상일"] = (now["t0"] + now["예측H"]).round().astype(int)
    print("\n[결핍 위험 일자 예측 - 100일 기준, 가까운 순 6건]")
    print(now.sort_values("예측H").head(6)[["unit", "item", "stock", "예측H", "결핍예상일", "H"]].rename(columns={"H": "실제H"}).round(1).to_string(index=False))

    # 가정 시나리오: 계획 템포 +30%와 다음 7일 악천후 50% 가정 시 예측 변화
    w = now.copy()
    w["plan7"] *= 1.3; w["plan_ratio"] *= 1.3; w["wx_fc"] = 0.5
    shift = np.clip(model.predict(w[X]), 1, None) - now["예측H"].values
    print(f"\n[가정 시나리오] 계획 템포 +30%, 악천후 50% 가정 시 예측 결핍 일수 평균 {shift.mean():+.2f}일 변화 (탄약 {shift[now['item'].values == '탄약'].mean():+.2f}일, 식량 {shift[now['item'].values == '식량'].mean():+.2f}일)")
    coef = pd.Series(model[-1].coef_, index=X).sort_values(key=abs, ascending=False)
    print("기준 모델 영향 큰 피처 상위 5:", {k: round(float(v), 2) for k, v in coef.head(5).items()})
    m = mape(y, p_ridge)
    print(f"\n목표(파생 피처 10개 이상, 기준 모델 MAPE 25% 이하): 파생 피처 {len(derived)}개 {'달성' if len(derived) >= 10 else '미달'}, MAPE {m:.1%} {'달성' if m <= 0.25 else '미달'}")


## 📋 3. 학습활동 4단계

### 활동 1: 무엇을 확인하고 계획하나요?

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, 작전 템포 변동과 기상 예보를 반영한 파생 피처를 활용할 때 계획 피처 제외 실험에서 무엇을 고정할 것인지 사전에 판단합니다.
  - 아울러 AI 에이전트에 위임할 분석 범위, 데이터 제약 사항, 입력 및 기간·단위·사용 가능 시점, 그리고 make_samples 함수의 입출력 계약 조건을 체계적으로 수립합니다.
- **수행 방식:**
  - 막연한 코딩이나 AI의 무분별한 답변에 의존하지 않고, 군수 데이터 분석 및 예측 모델의 신뢰성을 높이기 위한 명확한 사전 규칙과 검증 계획을 문서화합니다.

#### 2. 세부 수행 절차

- **원본 목표·기본 코드·자료 분석:**
  - 작전 템포 변동과 기상 예보를 반영한 파생 피처로 부대별 결핍 위험 일자(Stockout Date)를 예측하는 기준 모델을 구축하는 데 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다.
  - 계획 피처 제외 실험의 고정 변수 판단: 계획 피처(plan)를 제외하는 실험을 수행할 때, 모델의 공정하고 일관된 비교를 위해 어떤 조건과 파라미터를 고정해야 하는지 사전에 정의합니다.
- **make_samples 입출력 계약 및 AI 위임 범위 정의:**
  - 핵심 함수인 make_samples 의 입출력 계약 조건을 명확히 이해하고, AI 에이전트에 맡길 코드 구현 및 분석 범위, 데이터 무결성 검증 방법을 설정하여 검증 계획을 문서( forms/STR01_M04_forms.md )에 명시합니다.

#### 3. 주의 및 검증 사항

- 입력 시점이나 기간, 분모가 서로 다른 데이터를 동일한 조건으로 잘못 해석하지 않도록 데이터 해석의 엄정성을 유지해야 합니다.
- 본 단계에서 수립된 사전 판단과 검증 계획은 다음 단계(AI와 코드 구현 및 첫 성공 실행)의 기초가 되므로, 실제…

#### 완료 확인:

- 이 활동의 확인 근거 M04 계획: 작전 템포 변동과 기상 예보를 반영한 파생 피처로 부대별 결핍 위험 일자(Stockout Date)를 예측하는 기준 모델을 만든다.에 필요한 입력·기간·단위·사용 가능 시점과 make_samples의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2: AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 핵심 함수인 make_samples 를 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다.
  - 기간 설정 계약( d, lo=21, hi=125 )에 따라 과거 관측, 사전 계획, 가상 완전예보를 활용해 16개의 파생 피처와 메타데이터 H (향후 보급 없음 조건의 소진일수)를 생성합니다.
  - 이때 H는 모델 입력에 포함하지 않으며, t0 + H 가 학습 기준일 이내인 표본만 학습에 활용합니다.
  - 아울러 변경 과제( plan7 , plan_ratio 포함/제외 모델 비교)를 수행하여 파생 피처의 생성 시점과 사전 계획·완전예보 가정을 명확히 적용합니다.
- **수행 방식:**
  - AI와 협업하여 요구되는 피처 엔지니어링 계약 조건과 데이터 필터링 규칙을 실제 코드에 반영하고, 변경 전후의 실행 로그와 출력 결과물을 정확하게 확보합니다.

#### 2. 세부 수행 절차

- 핵심 함수 make_samples 구현 및 첫 성공 실행: 기간 파라미터( d, lo=21, hi=125 )를 준수하여 과거 관측 데이터, 사전 계획( plan ), 가상 완전예보( wx_fc )를 결합한 16개의 피처와 메타데이터 $H$를 산출하는 함수를 구현합니다.
- 소진일수 $H$는 향후 보급 중단 상황을 전제한 정답 값이므로 모델의 독립변수(입력)로 들어가지 않도록 엄격히 통제하고, t0 + H 가 학습 기준일 이내인 유효 표본만 추출하여 첫 성공 실행 로그와 결과를 보존합니다.
- 변경 과제 수행 ( plan7 , plan_ratio 포함/제외 모델 비교): 계획 피처인 plan7 과 plan_ratio 를 포함하는 모델과 제외하는 모델을 각각 구성하여, 동일한 학습 및 시험 기간 조건에서 예측 성능이 어떻게 달라지는지 비교 분석합니다.
- 파생 피처가 생성되는 시점과 사전 계획 및 완전예보 가정이 코드 내에서 어떻게 작동하는지 명…

#### 완료 확인:

- 이 활동의 확인 근거 M04 구현: src/features/str01_feature_engineering.py의 make_samples를저장하고계약〔d,lo=21,hi=125 → 과거 관측·사전 계획·가상 완전예보로 만든16개 피처와 메타데이터·H. H는 향후 보급 없음 조건의 소진일수이며 모델 입력에 넣지 않는다.
- t0+H가 학습 기준일 이내인 표본만 학습한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3: 원자료와 결과를 대조·수정하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 앞서 구현된 피처 엔지니어링 코드를 저장하고 재실행하여, 학습 기준일에 정답이 확정되지 않은 표본이 데이터셋에 섞이지 않았는지 엄밀히 검증합니다.
  - 특히 파생 피처 10개 이상을 생성하여 기준 모델 MAPE 25% 이하 를 달성하고, AI의 설명 및 코드 선택 3건 이상을 실제 군수 도메인 자료 및 실행 결과와 철저히 대조하여 보완된 답변을 도출합니다.
- **수행 방식:**
  - 단편적인 코드 실행에 그치지 않고, 학습 기준일 90일(t0+H≤90t_0 + H \le 90t0​+H≤90) 조건과 시험 기간(t0>90t_0 > 90t0​>90)의 분리 원칙을 적용하며 AI의 답변을 비판적으로 교차 검증합니다.

#### 2. 세부 수행 절차

- **수정 코드 저장 및 재실행:**
  - 피처 엔지니어링 및 모델링 코드를 안전하게 저장하고 재실행하여 에러 여부와 실행 로그를 확보합니다.
- **학습/시험 표본 분리 및 정답 확정 여부 검증:**
  - 학습 기준일에 정답이 확정된 표본(t0+H≤90t_0 + H \le 90t0​+H≤90)만 학습에 사용되었는지, 그리고 시험 데이터는 t0>90t_0 > 90t0​>90 조건으로 올바르게 분리되었는지 철저히 대조합니다.
  - 소진일수 H=0H=0H=0인 건은 예측 회귀의 분모에 포함되지 않으며 이미 결핍 규칙으로 분리되었는지 확인합니다.
  - AI 설명·코드 선택 3건 이상 근거 대조 및 보완: AI가 제안하거나 설명한 피처 설계 및 모델 선택 사항 중 핵심적인 3건 이상을 선정하여, 실제 군수 실무 데이터와 실행 결과에 비추어 타당한지 교차 검증하고 보완된 답변을 기록합니다.
  - plan (사전 계획 시나리오)과 wx_fc (가상 완전예보 가정)가 올바르게 반영되었는지 검토하되, 실제 미래 기상 실적을 현시점 입력으로 사용하는 오류를 범하지 않았는지 점검합니다.

#### 3. 주의 및 검증 사항

- 파생…

#### 완료 확인:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M04 검증: [선택] 파생 피처·기준 모델 — 파생 피처 10개 이상을 생성하고 기준 모델 MAPE 25% 이하를 달성했으며, 피처 설계가 군수 실무에서 납득 가능한 논리를 갖는가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - **수행 조건과 판정 기준 · 조건과 해석 범위**
    - 파생 피처 13개와 전체 입력 16개를 구분합니다.
    - 학습 기준일 90일까지 정답이 확정된〔t0+H≤90〕표본만 사용합니다.
    - 시험은 t0>90입니다.
  - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
    - H=0 건은 예측 회귀의 분모에 없고 이미 결핍 규칙으로 분리합니다.
    - plan은 미리 알고 있는 계획 시나리오, wx_fc는 원본의 완전예보 가정입니다.
    - 실제 미래 기상 실적을 현시점 입력으로 쓰는 검증으로 일반화하지 않습니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같…

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4: 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 미션 4(작전 템포 변동 및 기상 악화 연계 부대별 결핍 위험 일자 정밀 예측 고도화) 전 과정을 수행하며 산출된 피처 엔지니어링 소스 코드 , 피처 분석 보고서 , 예측 결과 데이터 파일 , 그리고 실습 기록 서식 을 누락 없이 집대성합니다.
  - 아울러 정량적 목표 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다.
- **수행 방식:**
  - 단편적인 파일 제출에 치중하지 않고, 이번 미션에서 구축한 파생 피처(13개 이상)와 기준 모델의 성능 지표(MAPE 25% 이하 달성 여부)를 객관적으로 검수하여 후속 미션(미션 5: 결핍 심각도 지수 산출 및 다단계 결핍 조기경보 엔진 개발)으로 온전히 인계할 수 있도록 패키지를 정비합니다.

#### 2. 세부 수행 절차

- **핵심 산출 파일 패키지 확정 및 제출:**
  - 피처 엔지니어링 소스 코드인 src/features/str01_feature_engineering.py , 피처 분석 보고서인 reports/str01_feature_report.md , 처리 결과 데이터인 outputs/M04/results.json , 그리고 실습 기록 서식인 forms/STR01_M04_forms.md 의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다.
- **코드 실행 결과 및 AI 대화 기록 연계 보존:**
  - make_samples 함수 구현 및 계획 피처( plan7 , plan_ratio ) 포함/제외 모델 비교 결과와, 분석 과정에서 AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 근거 대화 내용을 실습 대화 보존 위치에 명확히 연결합니다.
- **상태별 최종 판단 기록 작성:**
  - 정량적 목표 달성 여부, 현장 여건상 미처 측정하지 못한 영역, 문서 작성 완료 상태, 그리고 실제 교관 승인 대기 …

#### 완료 확인:

- 이 활동의 확인 근거 M04 완료: src/features/str01_feature_engineering.py, reports/str01_feature_report.md, outputs/M04/results.json, forms/STR01_M04_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 🔍 4. 평가 기준과 확인할 자료

- [정량적 평가] [선택] 파생 피처·기준 모델 — 파생 피처 10개 이상을 생성하고 기준 모델 MAPE 25% 이하를 달성했으며, 피처 설계가 군수 실무에서 납득 가능한 논리를 갖는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- **[정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.**
  - **PASS 증거**
    - **수행 조건과 판정 기준 · 조건과 해석 범위**
      - 파생 피처 13개와 전체 입력 16개를 구분합니다.
      - 학습 기준일 90일까지 정답이 확정된〔t0+H≤90〕표본만 사용합니다.
      - 시험은 t0>90입니다.
    - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
      - H=0 건은 예측 회귀의 분모에 없고 이미 결핍 규칙으로 분리합니다.
      - plan은 미리 알고 있는 계획 시나리오, wx_fc는 원본의 완전예보 가정입니다.
      - 실제 미래 기상 실적을 현시점 입력으로 쓰는 검증으로 일반화하지 않습니다.
    - **보완 방법**
      - 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- **[진행 추적 및 완료 기준] 정성·검증:**
  - **수행 조건과 판정 기준 · 조건과 해석 범위**
    - 파생 피처 13개와 전체 입력 16개를 구분합니다.
    - 학습 기준일 90일까지 정답이 확정된〔t0+H≤90〕표본만 사용합니다.
    - 시험은 t0>90입니다.
  - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
    - H=0 건은 예측 회귀의 분모에 없고 이미 결핍 규칙으로 분리합니다.
    - plan은 미리 알고 있는 계획 시나리오, wx_fc는 원본의 완전예보 가정입니다.
    - 실제 미래 기상 실적을 현시점 입력으로 쓰는 검증으로 일반화하지 않습니다.
  - **제출 파일 및 완료 기록**
    - 파일: src/features/str01_feature_engineering.py, reports/str01_feature_report.md, outputs/M04/results.json, forms/STR01_M04_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
    - 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
  - **보완 방법**
    - 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.
**실제 출력 없이 달성으로 표시하지 않습니다.**

In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 4
        DELIVERABLE_PATHS = ['src/features/str01_feature_engineering.py', 'reports/str01_feature_report.md', 'outputs/M04/results.json', 'forms/STR01_M04_forms.md']
        PROJECT_NAME = 'STR01'
        WORK_ROOT = Path(STR01_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                if state['loaded']:
                    state['dirty'] = True
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
